In [2]:
import pandas as pd

In [3]:
df=pd.read_csv('DataCoSupplyChainDataset.csv', encoding='ISO-8859-1')

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [6]:
columns_to_keep=['order date (DateOrders)','Days for shipping (real)', 'Days for shipment (scheduled)', 
    'Delivery Status', 'Late_delivery_risk',
    'Order Region', 'Order Country', 'Order City', 
    'Category Name', 'Product Name', 
    'Order Item Quantity', 'Sales per customer', 'Product Price', 'Order Status']

In [7]:
df_clean = df[columns_to_keep].copy()

In [8]:
df_clean.columns= df_clean.columns.str.replace(' ', '_').str.replace('(', '').str.replace(')', '').str.lower()

In [ ]:
df_clean.info()

In [10]:
df_clean['delay_in_days'] = df_clean['days_for_shipping_real'] - df_clean['days_for_shipment_scheduled']

In [ ]:
df_clean.head(5)

In [ ]:
print(df_clean[['days_for_shipping_real', 'days_for_shipment_scheduled', 'delay_in_days', 'delivery_status']].head())

In [13]:
df_clean['order_date'] = pd.to_datetime(df_clean['order_date_dateorders'])

In [ ]:
df_clean.info()

In [15]:
import sqlite3

In [ ]:
conn = sqlite3.connect(':memory:')
df_clean.to_sql('supply_chain', conn, index=False)

In [17]:
sql_query = """
WITH DailyRegionDelays AS (
    --Calculate the average delay per region for every single day
    SELECT 
        order_region,
        DATE(order_date) as order_day,
        AVG(delay_in_days) as avg_daily_delay
    FROM supply_chain
    WHERE delay_in_days > 0 -- We only want to analyze actual late shipments
    GROUP BY order_region, DATE(order_date)
)
--Use a Window Function to calculate the 30-day rolling average
SELECT 
    order_region,
    order_day,
    ROUND(avg_daily_delay, 2) as daily_delay,
    ROUND(AVG(avg_daily_delay) OVER (
        PARTITION BY order_region 
        ORDER BY order_day 
        ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ), 2) as rolling_30_day_delay
FROM DailyRegionDelays
ORDER BY order_region, order_day;
"""

In [18]:
rolling_delays_df = pd.read_sql(sql_query, conn)

In [19]:
rolling_delays_df.to_csv('Rolling_Delays.csv', index=False)